In [1]:
# Cell 1 — Imports and config
import sqlite3
import yaml
from openai import OpenAI
from oral_notes.s1_extract.doc_loader import GoogleDriveLoader
from oral_notes.s1_extract.text_extractor import TextExtractor
from config.config import OPENAI_API_KEY

DB_PATH              = "../DB/oedb_baseline_v3.db"
PROMPT_PATH_REFINER  = "../data/prompt_templates/refiner/prompt_refiner_ParReducer.yaml"
SERVICE_ACCOUNT_FILE = "../config/service_account_key.json"
LLM_MODEL            = "gpt-5.1"

client = OpenAI(api_key=OPENAI_API_KEY, base_url="https://llmproxy.uva.nl/v1")

In [7]:
# Cell 2 — Functions
def load_texts(notegroup_id):
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT note_url_QA, note_url_PARTICIPANT
            FROM notegroups
            WHERE notegroupID = ?
        """, (notegroup_id,))
        note_url_qa, note_url_participant = cursor.fetchone()

    file_loader = GoogleDriveLoader(SERVICE_ACCOUNT_FILE)
    extractor   = TextExtractor()

    all_texts = {}
    for label, url in [("QA", note_url_qa), ("PARTICIPANT", note_url_participant)]:
        if not url:
            continue
        result = file_loader.load(url)
        all_texts[label] = f"[Data source: {result['name']}]\n{extractor.extract(result)}"

    return all_texts


def run_refiner(notegroup_id, result_lastcall):
    all_texts = load_texts(notegroup_id)
    text_qa   = all_texts.get("QA", "")
    text_par  = all_texts.get("PARTICIPANT", "")

    prompts       = yaml.safe_load(open(PROMPT_PATH_REFINER))
    system_prompt = prompts["system"]
    user_prompt   = prompts["user"]["refine"].format(
        text_qa=text_qa,
        text_par=text_par,
        result_lastcall=result_lastcall
    )

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        seed=42,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
    )
    output = response.choices[0].message.content.strip()

    parts = output.split("###RESULT###")
    if len(parts) < 2:
        print("WARNING: ###RESULT### marker not found, keeping original result.")
        return result_lastcall

    result = parts[-1].strip()
    if result.lower() == "pass":
        print("Refiner returned: pass")
        return result_lastcall

    print("Refiner returned a correction.")
    return result

In [8]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 1
RESULT_LASTCALL = """[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | A.A |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | Y.A |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | L.H |
| Ahmad Noman | Male | Yemen | B1 | 24 | Helmond | A.N |
| Zaid Kurami | Male | Yemen | B1 | 25 | Hemlond | Z.K |
[/TABLE]"""

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: Rama - Note-taking 3 (1).docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Rama - Note-taking 3 (1).docx
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1

--- Refiner round 1 ---
Refiner returned a correction.

--- Refiner round 2 ---
Refiner returned: pass

=== Final refined result ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | null |
| Eyas | Female | Syria | B1 | 30 | Gemert | null |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | null |
| Nedal | Male | Syria | Z route, | 53 | Helmond | null |
| H

In [9]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 7
RESULT_LASTCALL = """[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 11 | Rawa Alshumry | Nesrine - Group Arabic 1 |  | R |
| 12 | Abdulaziz Al-Raimi | Danna - Group Arabic 2 |  | null |
| 14 | Abdullah Najjar | Danna - Group Arabic 2 |  | A.N |
| 16 | Victor | Danna - Group Arabic 2 |  | V |
[/TABLE]"""

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: Danna: Note-taking form 28.11 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Notes/Danna: Note-taking form 28.11
Loading: Deelnemers lijst + indeling #Session2 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 2/Deelnemers lijst + indeling #Session2

--- Refiner round 1 ---
Refiner returned a correction.

--- Refiner round 2 ---
Refiner returned a correction.

--- Refiner round 3 ---
Refiner returned a correction.

Reached max rounds (3) without pass.

=== Final refined result ===
[Data source: Deelnemers lijst + indeling #Session2.docx]
[TABLE]
|  | Name | Group | Present? | session_identifier |
| 11 | Rawa Alshumry | Nesrine - Group Arabic 1 |  | R |
| 12 | Abdulaziz Al-Raimi | Danna - Group Arabic 2 |  | A |
| 13 | Merry | Danna - Group Arabic 2 |  | null |
| 14 | Abdullah Najja

In [10]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 20
RESULT_LASTCALL = """[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت إلى هولندا؟ / Hangi koşullar altında Hollanda’ya geldin? | Welke route in de inburgering volg je? / Which route in integration are you following? / أي مسار في الاندماج تتبع؟ / Uyumda hangi yolu takip ediyorsun? | Wanneer ben je begonnen met de inburgering? / When did you start the integration process? / متى بدأت الاندماج؟ / Entegrasyon sürecine ne zaman başladın? | Waar woon je? / Where do you live? / أين تسكن؟ / Nerede yaşıyorsun? | Column 1 | session_identifier |
| Bevestigd |  | Iyad | TRUE | 10/21/2025 15:32:56 | Amer Al-maleki | 30 | Man / Man / رجل / Erkek | Jemen | Arabisch | Om asiel aan te vragen / To apply for asylum / لتقديم طلب لجوء / İltica başvurusu yapmak için | B1-route | 12/26/2022 | Monnickendam |  | A |
| Bevestigd |  | Iyad | TRUE | 11/11/2025 12:19:45 | ‪Reem Al abbas‬‏ | 19 | Vrouw / Woman / امرأة / Kadın | Syrië | Arabisch | Om te herenigen met een gezinslid die asiel heeft aangevraagd (gezinshereniging) / To reunite with a family member who has applied for asylum / للالتحاق بأحد أفراد الأسرة الذي قدّم طلب لجوء / İltica başvurusunda bulunmuş bir aile üyesiyle birleşmek için | B1-route | 5/25/2024 | MONNICKENDAM |  | R |
| Bevestigd |  | Iyad | TRUE | 11/7/2025 20:36:16 | Mohammed Alnazar | 42 | Man / Man / رجل / Erkek | Iraq | Arabic | Om asiel aan te vragen / To apply for asylum / لتقديم طلب لجوء / İltica başvurusu yapmak için | B1-route | 7/17/2025 | Landsmeer |  | M |
| Bevestigd |  | Iyad | TRUE | 11/14/2025 10:37:41 | Riham almishael | 35 | Vrouw / Woman / امرأة / Kadın | سوريا | عربي | Om te herenigen met een gezinslid die asiel heeft aangevraagd (gezinshereniging) / To reunite with a family member who has applied for asylum / للالتحاق بأحد أفراد الأسرة الذي قدّم طلب لجوء / İltica başvurusunda bulunmuş bir aile üyesiyle birleşmek için | Onderwijsroute | 11/14/2025 | MonnickendamCornelis Dirkszoonlaan 316 |  | null |
| Bevestigd |  | Iyad | TRUE | 11/9/2025 1:32:23 | maha | 30 | Vrouw / Woman / امرأة / Kadın | Serya | Koerdisch Arabisch | Om asiel aan te vragen / To apply for asylum / لتقديم طلب لجوء / İltica başvurusu yapmak için | B1-route | 10/29/2025 | Waterland |  | M.H |
| Bevestigd |  | Iyad | TRUE | 11/14/2025 11:43:34 | Shirin Hamdou | 36 | Non-binair / Gender-niet-conformerend / Non-binary / Gender non-conforming / غير ثنائي / غير ممتثل للجندر / İkili olmayan / Toplumsal cinsiyet normlarına uymayan | سوريا | كردي | Om te herenigen met een gezinslid die asiel heeft aangevraagd (gezinshereniging) / To reunite with a family member who has applied for asylum / للالتحاق بأحد أفراد الأسرة الذي قدّم طلب لجوء / İltica başvurusunda bulunmuş bir aile üyesiyle birleşmek için | B1-route | 8/25/2025 | Monnickendam |  | Sh |
"""

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: NOTITIES_IYAD (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_IYAD
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)

--- Refiner round 1 ---
Refiner returned a correction.

--- Refiner round 2 ---
Refiner returned a correction.

--- Refiner round 3 ---
Refiner returned a correction.

Reached max rounds (3) without pass.

=== Final refined result ===
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ Wh

In [4]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 21
RESULT_LASTCALL =\
    """
[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cinsiyetin ne? | Wat is jouw land van herkomst? / What is your country of origin? / ما بلدُك الأصلي؟ / Memleketin neresi? | Wat is jouw voorkeurstaal? / What is your preferred language? / ما هي لغتك المفضلة؟ / Tercih ettiğin dil nedir? | Onder welke omstandigheden ben je naar Nederland gekomen? / Under what circumstances did you come to The Netherlands? / تحت أي ظروف أتيت إلى هولندا؟ / Hangi koşullar altında Hollanda’ya geldin? | Welke route in de inburgering volg je? / Which route in integration are you following? / أي مسار في الاندماج تتبع؟ / Uyumda hangi yolu takip ediyorsun? | Wanneer ben je begonnen met de inburgering? / When did you start the integration process? / متى بدأت الاندماج؟ / Entegrasyon sürecine ne zaman başladın? | Waar woon je? / Where do you live? / أين تسكن؟ / Nerede yaşıyorsun? | Column 1 | session_identifier |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Bevestigd |  | Floris/Anne | TRUE | 11/11/2025 13:41:03 | Musfira Mahnoor | 23 | Vrouw / Woman / امرأة / Kadın | Pakistan | English | Om te herenigen met een gezinslid die asiel heeft aangevraagd (gezinshereniging) / To reunite with a family member who has applied for asylum / للالتحاق بأحد أفراد الأسرة الذي قدّم طلب لجوء / İltica başvurusunda bulunmuş bir aile üyesiyle birleşmek için | Onderwijsroute | 7/25/2025 | Ilpendam | 1,5 jaar in Waterland | Vrouw, 23, B1 |
| Bevestigd |  | Floris/Anne | TRUE | 11/7/2025 15:09:22 | Angela Paola Barragán sanchez | 42 | Vrouw / Woman / امرأة / Kadın | ColombiaC | Español | Als een familiemigrant, om te herenigen met een familielid die ingezetene is in Nederland / As a family migrant, to reunite with a relative who is a resident in the Netherlands / كمهاجر عائلي، للالتحاق بأحد الأقارب المُقيمين في هولندا / Aile göçmeni olarak, Hollanda’da ikamet eden bir aile üyesiyle birleşmek için | B1-route | 11/14/2022 | Monnickendam | 3 jaar, 4 maand in Waterland | Vrouw, 42, B1 |
    """

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: NOTITIES_Floris_en_Anne.docx (application/vnd.openxmlformats-officedocument.wordprocessingml.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Groep_gesprek_met_deelnemers/Notes/NOTITIES_Floris_en_Anne.docx
Loading: Perspectief inburgeraars Waterland/Landsmeer (Responses) (application/vnd.google-apps.spreadsheet)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1380 Perspectief Inburgeraars Waterland_Landsmeer/Werving en participants/Perspectief inburgeraars Waterland/Landsmeer (Responses)
Refiner returned: pass

=== Final refined result ===

[Data source: Perspectief inburgeraars Waterland/Landsmeer (Responses).csv]
| Intake status | Beschikbaarheid | Interviewer | Aanwezig! | Timestamp | Wat is je naam? / What is your name? / ما اسمك؟ / Adın ne? | Wat is je leeftijd?/ What is your age? / كم عمرك؟ / Kaç yaşındasın? | Wat is je gender? / What is your gender? / ما هو نوعك الاجتماعي؟ / Cin

In [5]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 3
RESULT_LASTCALL =\
    """
[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | M.A. |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | A.A. |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | A.A.2 |
| Lydia | Danna - Group Arabic 1 |  | L |
[/TABLE]
    """

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: Note form Danna 22 Nov (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Notes/Note form Danna 22 Nov
Loading: Deelnemers lijst + indeling (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/   ARCHIEF /1374 | Wegen naar Werk/Expertpools/Session 1/Deelnemers lijst + indeling
Refiner returned: pass

=== Final refined result ===

[Data source: Deelnemers lijst + indeling.docx]
[TABLE]
| Name | Group | Present? | session_identifier |
| Mortada Abu Hassan | Danna - Group Arabic 1 |  | M.A. |
| Ahmad Alhussein Alsatouf | Danna - Group Arabic 1 |  | A.A. |
| Alaa Abdal Wahab | Danna - Group Arabic 1 |  | A.A.2 |
| Lydia | Danna - Group Arabic 1 |  | L |
[/TABLE]
    


In [11]:
# Cell 3 — Run (adjust parameters here)
NOTEGROUP_ID = 2
RESULT_LASTCALL =\
    """
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | Kh |
| Eyas | Female | Syria | B1 | 30 | Gemert | Eyas |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | Ahmad |
| Nedal | Male | Syria | Z route, | 53 | Helmond | N |
| Hassan | Male | Syria | Z route, | 30 | Helmond | H |
| Wasim | Male | Syria | Zroute, | 53 | Helmond | Wasem |
| Firas Aljnainati | Male | Syria, | B1 | 32 | Helmond | Feras |
| Ahmad Ahmad | Male | Syria | B1 | 35 | Helmond | null |
| Yasmine Ahmad | Female | Syria | B1 | 40 | Helmond | null |
| Layla Hamliko | Female | Syria | B1 | 50 | Gemert | null |
[/TABLE]
    """

final_result = run_refiner(NOTEGROUP_ID, RESULT_LASTCALL)

print("\n=== Final refined result ===")
print(final_result)

Loading: Iyad - Note-taking 3.12 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Iyad - Note-taking 3.12
Loading: Aanwezig sessie 1 (application/vnd.google-apps.document)
Drive path: 01 extern/01 huidige klanten : projecten/1306 - M&E Helmond de Peel Regio/4. Expertpool _ klankbordgroep /Notities en analyse /Aanwezig sessie 1

--- Refiner round 1 ---
Refiner returned a correction.

--- Refiner round 2 ---
Refiner returned a correction.

--- Refiner round 3 ---
Refiner returned a correction.

Reached max rounds (3) without pass.

=== Final refined result ===
[Data source: Aanwezig sessie 1.docx]
[TABLE]
| Naam | Gender | Country | Route | Leeftijd | Gemeente | session_identifier |
| Khetam | Female | Syria | Z route | 43 | Helmond | Kh |
| Eyas | Female | Syria | B1 | 30 | Gemert | Eyas |
| Ahmad Brimo | Male | Syria | Z route, | 51 | Gemert | A |
| Nedal | M